# Example 6-7: Performing a Combined Maneuver
### _Fundamentals of Astrodynamics and Applications_, 5th Ed., 2022, pp. 360-361

This notebook demonstrates the process to find the change in velocity for a combined maneuver.

## Install and Import Libraries
---

First, install `valladopy` if it doesn't already exist in your environment:

In [1]:
!pip install -r ../valladopy_version.txt

Import the relevant `valladopy` modules:

In [2]:
import numpy as np
import valladopy.constants as const
from valladopy.astro.maneuver.transfer import combined

## Problem Definition
---

GIVEN:&ensp; $i_{initial}$ = 28.5°, $i_{final}$ = 0°, $e_{initial}$ = $e_{final}$ = 0.0, 
$alt_{initial}$ = 191 km, $alt_{final}$ = 35,780 km<br>
FIND: &emsp;$\Delta{i_{initial}}$, $\Delta{v_a}$, $\Delta{v_b}$, $\tau_{trans}$, $\gamma_a$, $\gamma_b$

In [3]:
i_init = np.radians(28.5)  # rad
i_final = 0                # rad
e_init = 0
alt_init = 191             # km
alt_final = 35780          # km

## Solution
---

Begin by determining radii values:

In [4]:
r_init = const.RE + alt_init
r_final = const.RE + alt_final

print(f'r_initial:\t{r_init}\tkm\t')
print(f'r_final:\t{r_final}\tkm\t')

r_initial:	6569.1363	km	
r_final:	42158.1363	km	


Then we determine the velocities with:

$$
\begin{aligned}
v_{initial} &= \sqrt{\frac{\mu}{r_{initial}}} \\
v_{final} &= \sqrt{\frac{\mu}{r_{final}}} \\
v_{trans_a} &= \sqrt{\frac{2\mu}{r_{initial}} - \frac{\mu}{a_{trans}}} \\
v_{trans_b} &= \sqrt{\frac{2\mu}{r_{final}} - \frac{\mu}{a_{trans}}}
\end{aligned}
$$

The change in velocities is then:

$$
\begin{aligned}
\Delta{v_a} &= \sqrt{v_{initial}^2 + v_{trans_a}^2 - 2 \ v_{initial} \ v_{trans_a} \cos(\Delta{i_1})} \\
\Delta{v_b} &= \sqrt{v_{final}^2 + v_{trans_b}^2 - 2 \ v_{final} \ v_{trans_b} \cos(\Delta{i_2})}
\end{aligned}
$$

Determine how much to change the inclination with the first burn with **Algorithm 42**.

The orbital ratio is defined as:

$$
R = \frac{r_{final}}{r_{initial}}
$$

Then:
$$
\begin{aligned}
s &\cong \frac{1}{\Delta{i}} \tan^{-1} \left( \frac{\sin(\Delta{i})}{R^{3/2} + \cos(\Delta{i})} \right) \\
\Delta{i_1} &= s\Delta{i} \\
\Delta{i_2} &= (1 - s) \Delta{i}
\end{aligned}
$$

We can then use **Algorithm 43** to determine the orientation (payload angles) of the burns:

$$
\begin{aligned}
\cos(\gamma_a) &= -\frac{v_{initial}^2 + \Delta v_a^2 - v_{trans_a}^2}{2 \ v_{initial}\ \Delta v_a} \\
\\
\cos(\gamma_b) &= -\frac{v_{trans_b}^2 + \Delta v_b^2 - v_{final}^2}{2 \ v_{trans_b}\ \Delta v_b}
\end{aligned}
$$

This can all be done with the `combined` routine:

In [5]:
delta_i1, delta_i2, delta_va, delta_vb, dtsec, gam_a, gam_b = combined(
    r_init, r_final, e_init, nuinit=0, deltai=(i_final - i_init)
)

print(f'Δi_1:\t{np.degrees(delta_i1):.6f}\tdeg')
print(f'Δi_2:\t{np.degrees(delta_i2):.6f}\tdeg\n')
print(f'Δv_a:\t{delta_va:.6f}\tkm/s')
print(f'Δv_b:\t{delta_vb:.6f}\tkm/s\n')
print(f'γ_a:\t{np.degrees(gam_a):.4f}\t\tdeg')
print(f'γ_b:\t{np.degrees(gam_b):.4f}\t\tdeg')

Δi_1:	-1.594962	deg
Δi_2:	-26.905038	deg

Δv_a:	2.469670	km/s
Δv_b:	1.802216	km/s

γ_a:	6.6315		deg
γ_b:	50.5395		deg
